CKD NB3: Trustworthiness, Research Synthesis and Reproducibility

Project: Trustworthy Agentic AI for Healthcare

Purpose

This notebook examines the trustworthiness, interpretability, robustness, research implications and reproducibility of the Chronic Kidney Disease (CKD) modelling workflow developed in NB1 and evaluated in NB2.

Research objectives

1. Assess the strengths and limitations of the CKD modelling workflow.
2. Synthesize model performance, coefficient interpretation, calibration and bootstrap stability findings.
3. Discuss missing data, class imbalance, potential bias and clinical limitations.
4. Document the workflow to support reproducibility and future research.
5. Identify improvements required before any clinical application.

Target definition and clinical limitation

The target is a cross-sectional kidney-abnormality research indicator derived from eGFR and urine albumin-to-creatinine ratio criteria. It is not a confirmed clinical CKD diagnosis and does not represent prediction of future disease.

Planned demonstrator integration

The CKD research component will later be integrated into the existing healthcare AI demonstrator alongside the diabetes component. Claude AI will be used to assist with this integration, following the approach used for the diabetes demonstrator. Any displayed predictions and explanations must communicate the research limitations clearly and must not be presented as a clinical diagnosis.

Notebook structure

1. Trustworthiness and limitations
2. Explainability and coefficient stability synthesis
3. Robustness and reliability considerations
4. Research findings and literature synthesis
5. Reproducibility documentation
6. Conclusions and future work

In [1]:
# Cell 1: Environment and reproducibility setup

import sys
import platform
import numpy as np
import pandas as pd
import sklearn
from datetime import datetime, timezone

# Reproducibility settings
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Project: Trustworthy Agentic AI for Healthcare")
print("Notebook: NB3 - Trustworthiness, Research Synthesis and Reproducibility")
print("Python version:", sys.version.split()[0])
print("Platform:", platform.platform())
print("NumPy version:", np.__version__)
print("Pandas version:", pd.__version__)
print("Scikit-learn version:", sklearn.__version__)
print("Random state:", RANDOM_STATE)
print("Execution timestamp (UTC):", datetime.now(timezone.utc).isoformat())
print("\nNB3 environment setup completed.")

Project: Trustworthy Agentic AI for Healthcare
Notebook: NB3 - Trustworthiness, Research Synthesis and Reproducibility
Python version: 3.13.16
Platform: Linux-6.6.122+-x86_64-with-glibc2.39
NumPy version: 2.1.3
Pandas version: 2.2.3
Scikit-learn version: 1.6.1
Random state: 42
Execution timestamp (UTC): 2026-10-15T20:16:38.574229+00:00

NB3 environment setup completed.


In [2]:
# Cell 2: Record verified study facts and NB2 evaluation results

study_facts = {
    "data_source": "NHANES 2017–March 2020 Pre-Pandemic",
    "adult_cohort_size": 9693,
    "labelled_participants": 8267,
    "unlabelled_participants": 1426,
    "labelled_negative": 6644,
    "labelled_positive": 1623,
    "heldout_test_size": 1654,
    "heldout_test_negative": 1329,
    "heldout_test_positive": 325,
    "candidate_predictors": 15,
    "transformed_features": 28,
    "bootstrap_fits": 30,
    "random_state": RANDOM_STATE,
    "target_definition": (
        "Cross-sectional kidney-abnormality research indicator based on "
        "eGFR and urine albumin-to-creatinine ratio criteria"
    ),
}

nb2_test_metrics = pd.DataFrame([
    {
        "model": "Standard Logistic Regression",
        "accuracy": 0.8174,
        "precision": 0.6162,
        "recall": 0.1877,
        "f1": 0.2877,
        "roc_auc": 0.7994,
        "average_precision": 0.5084,
        "brier_score": 0.1259,
        "log_loss": 0.4009,
    },
    {
        "model": "Class-weighted Logistic Regression",
        "accuracy": 0.7152,
        "precision": 0.3866,
        "recall": 0.7662,
        "f1": 0.5139,
        "roc_auc": 0.7994,
        "average_precision": 0.5041,
        "brier_score": 0.1867,
        "log_loss": 0.5567,
    },
])

assert (
    study_facts["labelled_negative"] + study_facts["labelled_positive"]
    == study_facts["labelled_participants"]
)
assert (
    study_facts["heldout_test_negative"] + study_facts["heldout_test_positive"]
    == study_facts["heldout_test_size"]
)
assert len(nb2_test_metrics) == 2
assert nb2_test_metrics["model"].is_unique

print("Study facts and NB2 results recorded successfully.")
print("\nCohort summary:")
for key in [
    "adult_cohort_size",
    "labelled_participants",
    "unlabelled_participants",
    "labelled_negative",
    "labelled_positive",
    "heldout_test_size",
    "candidate_predictors",
    "transformed_features",
    "bootstrap_fits",
]:
    print(f"{key}: {study_facts[key]}")

print("\nNB2 held-out test results:")
display(nb2_test_metrics.round(4))

Study facts and NB2 results recorded successfully.

Cohort summary:
adult_cohort_size: 9693
labelled_participants: 8267
unlabelled_participants: 1426
labelled_negative: 6644
labelled_positive: 1623
heldout_test_size: 1654
candidate_predictors: 15
transformed_features: 28
bootstrap_fits: 30

NB2 held-out test results:


,model,accuracy,precision,recall,f1,roc_auc,average_precision,brier_score,log_loss
0,Standard Logistic Regression,0.8174,0.6162,0.1877,0.2877,0.7994,0.5084,0.1259,0.4009
1,Class-weighted Logistic Regression,0.7152,0.3866,0.7662,0.5139,0.7994,0.5041,0.1867,0.5567


In [3]:
# Cell 3: Compare model trade-offs

metric_directions = {
    "accuracy": "higher is better",
    "precision": "higher is better",
    "recall": "higher is better",
    "f1": "higher is better",
    "roc_auc": "higher is better",
    "average_precision": "higher is better",
    "brier_score": "lower is better",
    "log_loss": "lower is better",
}

standard = nb2_test_metrics.set_index("model").loc[
    "Standard Logistic Regression"
]
weighted = nb2_test_metrics.set_index("model").loc[
    "Class-weighted Logistic Regression"
]

comparison_rows = []

for metric, direction in metric_directions.items():
    standard_value = standard[metric]
    weighted_value = weighted[metric]

    comparison_rows.append({
        "metric": metric,
        "standard_model": standard_value,
        "weighted_model": weighted_value,
        "weighted_minus_standard": weighted_value - standard_value,
        "preferred_value_direction": direction,
    })

tradeoff_table = pd.DataFrame(comparison_rows)

print("NB2 held-out test trade-off comparison")
display(tradeoff_table.round(4))

print("\nInterpretation:")
print("- The weighted model identifies more positive cases at threshold 0.5.")
print("- The standard model has higher precision and lower probability-error scores.")
print("- Both models have the same ROC-AUC in the recorded test results.")
print("- These comparisons are descriptive; they do not establish clinical suitability.")

NB2 held-out test trade-off comparison


,metric,standard_model,weighted_model,weighted_minus_standard,preferred_value_direction
0,accuracy,0.8174,0.7152,-0.1022,higher is better
1,precision,0.6162,0.3866,-0.2296,higher is better
2,recall,0.1877,0.7662,0.5785,higher is better
3,f1,0.2877,0.5139,0.2262,higher is better
4,roc_auc,0.7994,0.7994,0.0000,higher is better
5,average_precision,0.5084,0.5041,-0.0043,higher is better
6,brier_score,0.1259,0.1867,0.0608,lower is better
7,log_loss,0.4009,0.5567,0.1558,lower is better



Interpretation:
- The weighted model identifies more positive cases at threshold 0.5.
- The standard model has higher precision and lower probability-error scores.
- Both models have the same ROC-AUC in the recorded test results.
- These comparisons are descriptive; they do not establish clinical suitability.


In [4]:
# Cell 4: Document trustworthiness risks and mitigation strategies

trustworthiness_risks = pd.DataFrame([
    {
        "risk": "Cross-sectional target limitation",
        "evidence_or_concern": (
            "The target represents kidney-abnormality criteria at one survey period, "
            "not confirmed clinical CKD or future disease."
        ),
        "mitigation": (
            "Use research-oriented wording; require clinical validation before "
            "diagnostic use."
        ),
    },
    {
        "risk": "Class imbalance",
        "evidence_or_concern": (
            "The labelled cohort contains 1623 positive and 6644 negative cases."
        ),
        "mitigation": (
            "Report precision, recall, F1, specificity, and precision-recall "
            "metrics alongside accuracy."
        ),
    },
    {
        "risk": "False negatives and false positives",
        "evidence_or_concern": (
            "At threshold 0.5, the standard model has 264 false negatives; "
            "the weighted model has 395 false positives on the held-out test set."
        ),
        "mitigation": (
            "Evaluate threshold choices using training-only validation and "
            "assess the consequences with clinical experts."
        ),
    },
    {
        "risk": "Missing data",
        "evidence_or_concern": (
            "Some predictors have missing values, including blood-pressure "
            "measurements."
        ),
        "mitigation": (
            "Keep imputation within the training pipeline and examine missingness "
            "patterns and sensitivity."
        ),
    },
    {
        "risk": "Calibration",
        "evidence_or_concern": (
            "The weighted model has a higher Brier score and log loss than "
            "the standard model in the recorded test results."
        ),
        "mitigation": (
            "Assess calibration on independent data; do not interpret scores "
            "as clinical probabilities without validation."
        ),
    },
    {
        "risk": "Generalizability and bias",
        "evidence_or_concern": (
            "Results come from a specific NHANES survey cohort and may not "
            "generalize to other populations or healthcare settings."
        ),
        "mitigation": (
            "Conduct subgroup and external validation before considering "
            "real-world use."
        ),
    },
    {
        "risk": "Explanation stability",
        "evidence_or_concern": (
            "Only 30 bootstrap refits were completed, and some coefficients "
            "showed unstable signs."
        ),
        "mitigation": (
            "Present explanations cautiously and repeat stability analysis "
            "with more resamples and independent data."
        ),
    },
    {
        "risk": "Demonstrator overclaiming",
        "evidence_or_concern": (
            "Users may mistake a model output or AI-generated explanation "
            "for a medical diagnosis."
        ),
        "mitigation": (
            "Clearly label outputs as research demonstrations, show limitations, "
            "and avoid autonomous diagnostic or treatment advice."
        ),
    },
])

print("Trustworthiness risk register created.")
print("Risks documented:", len(trustworthiness_risks))
display(trustworthiness_risks)

Trustworthiness risk register created.
Risks documented: 8


,risk,evidence_or_concern,mitigation
0,Cross-sectional target limitation,The target represents kidney-abnormality crite...,Use research-oriented wording; require clinica...
1,Class imbalance,The labelled cohort contains 1623 positive and...,"Report precision, recall, F1, specificity, and..."
2,False negatives and false positives,"At threshold 0.5, the standard model has 264 f...",Evaluate threshold choices using training-only...
3,Missing data,"Some predictors have missing values, including...",Keep imputation within the training pipeline a...
4,Calibration,The weighted model has a higher Brier score an...,Assess calibration on independent data; do not...
5,Generalizability and bias,Results come from a specific NHANES survey coh...,Conduct subgroup and external validation befor...
6,Explanation stability,"Only 30 bootstrap refits were completed, and s...",Present explanations cautiously and repeat sta...
7,Demonstrator overclaiming,Users may mistake a model output or AI-generat...,Clearly label outputs as research demonstratio...


In [5]:
# Cell 5: Explainability and coefficient stability synthesis

explainability_summary = pd.DataFrame([
    {
        "finding": "Age",
        "evidence": "Coefficient +0.6065; sign stability 100%",
        "interpretation": (
            "Age had a positive association with the model's predicted "
            "positive class, conditional on the other encoded predictors."
        ),
        "caution": "Model association, not causation."
    },
    {
        "finding": "Self-reported health category",
        "evidence": (
            "HUQ010_5.0 coefficient +0.5373; bootstrap sign stability 100%"
        ),
        "interpretation": (
            "This encoded health-status category had a positive fitted "
            "coefficient in the weighted model."
        ),
        "caution": (
            "Category coding and reference levels affect interpretation."
        )
    },
    {
        "finding": "Diabetes questionnaire category",
        "evidence": (
            "DIQ010_1.0 coefficient +0.4244; bootstrap sign stability 100%"
        ),
        "interpretation": (
            "This encoded diabetes-response category was positively associated "
            "with the model score."
        ),
        "caution": "Not proof of an independent causal effect."
    },
    {
        "finding": "Systolic blood pressure",
        "evidence": "Standardized coefficient +0.2712; sign stability 100%",
        "interpretation": (
            "Higher standardized systolic blood pressure was associated "
            "with a higher model score."
        ),
        "caution": "Association within this dataset and fitted model only."
    },
    {
        "finding": "Glycated haemoglobin",
        "evidence": "Standardized coefficient +0.2477; sign stability 100%",
        "interpretation": (
            "Higher standardized LBXGH was associated with a higher model score."
        ),
        "caution": "Not a clinical threshold or causal conclusion."
    },
    {
        "finding": "BMI and waist circumference",
        "evidence": "Correlation r = 0.906",
        "interpretation": (
            "These numeric predictors are strongly correlated, which can "
            "complicate the interpretation of individual coefficients."
        ),
        "caution": "Correlation alone does not require removing a predictor."
    },
    {
        "finding": "Less stable coefficients",
        "evidence": (
            "Some bootstrap sign-stability estimates were approximately "
            "53%–73% across 30 refits."
        ),
        "interpretation": (
            "The direction of some fitted categorical associations varied "
            "across bootstrap samples."
        ),
        "caution": (
            "Thirty refits provide exploratory evidence, not definitive "
            "uncertainty estimates."
        )
    },
])

print("Explainability synthesis created.")
display(explainability_summary)

print("\nBootstrap stability overview:")
print("Bootstrap refits:", study_facts["bootstrap_fits"])
print("Transformed features:", study_facts["transformed_features"])
print("Features with 100% sign stability among the reported summary:", 14)
print("Features stable in sign at least 90% of refits:", 19)
print("\nInterpret coefficients as model associations, not causal effects.")

Explainability synthesis created.


,finding,evidence,interpretation,caution
0,Age,Coefficient +0.6065; sign stability 100%,Age had a positive association with the model'...,"Model association, not causation."
1,Self-reported health category,HUQ010_5.0 coefficient +0.5373; bootstrap sign...,This encoded health-status category had a posi...,Category coding and reference levels affect in...
2,Diabetes questionnaire category,DIQ010_1.0 coefficient +0.4244; bootstrap sign...,This encoded diabetes-response category was po...,Not proof of an independent causal effect.
3,Systolic blood pressure,Standardized coefficient +0.2712; sign stabili...,Higher standardized systolic blood pressure wa...,Association within this dataset and fitted mod...
4,Glycated haemoglobin,Standardized coefficient +0.2477; sign stabili...,Higher standardized LBXGH was associated with ...,Not a clinical threshold or causal conclusion.
5,BMI and waist circumference,Correlation r = 0.906,These numeric predictors are strongly correlat...,Correlation alone does not require removing a ...
6,Less stable coefficients,Some bootstrap sign-stability estimates were a...,The direction of some fitted categorical assoc...,"Thirty refits provide exploratory evidence, no..."



Bootstrap stability overview:
Bootstrap refits: 30
Transformed features: 28
Features with 100% sign stability among the reported summary: 14
Features stable in sign at least 90% of refits: 19

Interpret coefficients as model associations, not causal effects.


In [7]:
# Cell 6: Reproducibility and data-governance checklist

reproducibility_checklist = pd.DataFrame([
    {
        "item": "Data provenance",
        "status": "Documented",
        "detail": "NHANES 2017–March 2020 Pre-Pandemic public-use data."
    },
    {
        "item": "Cohort definition",
        "status": "Documented",
        "detail": "Adults aged 18 years or older; 9,693 participants."
    },
    {
        "item": "Target construction",
        "status": "Documented",
        "detail": "Cross-sectional eGFR/UACR-based research indicator."
    },
    {
        "item": "Leakage prevention",
        "status": "Documented in NB1/NB2",
        "detail": "Direct target-defining variables and markers excluded from predictors."
    },
    {
        "item": "Train/test split",
        "status": "Documented in NB1/NB2",
        "detail": "Stratified 80/20 split with random_state=42."
    },
    {
        "item": "Preprocessing",
        "status": "Documented in NB1/NB2",
        "detail": "Imputation, scaling and one-hot encoding within model pipelines."
    },
    {
        "item": "Evaluation",
        "status": "Documented in NB2",
        "detail": "Held-out metrics, confusion matrices, calibration-related metrics and threshold analysis."
    },
    {
        "item": "Coefficient stability",
        "status": "Exploratory",
        "detail": "30 bootstrap refits; more resampling may be needed."
    },
    {
        "item": "External validation",
        "status": "Not completed",
        "detail": "Independent external cohort evaluation is still required."
    },
    {
        "item": "Clinical validation",
        "status": "Not completed",
        "detail": "No clinical deployment or diagnostic validation has been established."
    },
    {
        "item": "Software environment",
        "status": "Recorded in NB3",
        "detail": "Python, NumPy, pandas, scikit-learn and random seed."
    },
    {
        "item": "Demonstrator safety",
        "status": "Planned",
        "detail": "Claude-assisted CKD integration must preserve research limitations and avoid diagnostic claims."
    },
])

print("Reproducibility and data-governance checklist created.")
display(reproducibility_checklist)

print("\nChecklist items:", len(reproducibility_checklist))
print("External validation completed: No")
print("Clinical validation completed: No")
print("Claude-assisted CKD demonstrator integration: Planned")

Reproducibility and data-governance checklist created.


,item,status,detail
0,Data provenance,Documented,NHANES 2017–March 2020 Pre-Pandemic public-use...
1,Cohort definition,Documented,"Adults aged 18 years or older; 9,693 participa..."
2,Target construction,Documented,Cross-sectional eGFR/UACR-based research indic...
3,Leakage prevention,Documented in NB1/NB2,Direct target-defining variables and markers e...
4,Train/test split,Documented in NB1/NB2,Stratified 80/20 split with random_state=42.
5,Preprocessing,Documented in NB1/NB2,"Imputation, scaling and one-hot encoding withi..."
6,Evaluation,Documented in NB2,"Held-out metrics, confusion matrices, calibrat..."
7,Coefficient stability,Exploratory,30 bootstrap refits; more resampling may be ne...
8,External validation,Not completed,Independent external cohort evaluation is stil...
9,Clinical validation,Not completed,No clinical deployment or diagnostic validatio...



Checklist items: 12
External validation completed: No
Clinical validation completed: No
Claude-assisted CKD demonstrator integration: Planned


In [8]:
# Cell 7: Synthesize research findings and limitations

research_findings = [
    {
        "area": "Discrimination",
        "finding": (
            "Both models recorded ROC-AUC of 0.7994 on the held-out test set."
        ),
        "limitation": (
            "This measures ranking discrimination, not clinical usefulness "
            "or diagnostic accuracy."
        ),
    },
    {
        "area": "Positive-case detection",
        "finding": (
            "The class-weighted model recorded recall 0.7662 and F1 0.5139 "
            "at threshold 0.5."
        ),
        "limitation": (
            "Its precision was 0.3866, and its specificity was 0.7028."
        ),
    },
    {
        "area": "Conservative positive predictions",
        "finding": (
            "The standard model recorded precision 0.6162 and specificity 0.9714 "
            "at threshold 0.5."
        ),
        "limitation": (
            "Its recall was only 0.1877, so many labelled positive cases "
            "were missed."
        ),
    },
    {
        "area": "Probability quality",
        "finding": (
            "The standard model had lower Brier score (0.1259) and log loss "
            "(0.4009) than the weighted model."
        ),
        "limitation": (
            "These results do not establish reliable clinical probabilities "
            "or guarantee calibration in another population."
        ),
    },
    {
        "area": "Explainability",
        "finding": (
            "Coefficient analysis and bootstrap refits provide exploratory "
            "information about model associations and sign stability."
        ),
        "limitation": (
            "Coefficients are not causal effects; 30 bootstrap refits are "
            "limited evidence of stability."
        ),
    },
    {
        "area": "Clinical interpretation",
        "finding": (
            "The workflow demonstrates modelling and evaluation of a "
            "cross-sectional kidney-abnormality research indicator."
        ),
        "limitation": (
            "It is not a confirmed CKD diagnosis or a model of future disease."
        ),
    },
]

research_findings_df = pd.DataFrame(research_findings)

print("Research synthesis completed.")
display(research_findings_df)

print(
    "\nOverall conclusion: The results demonstrate a trade-off between "
    "positive-case detection and precision. Further independent validation "
    "and clinical assessment are required before any clinical use."
)

Research synthesis completed.


,area,finding,limitation
0,Discrimination,Both models recorded ROC-AUC of 0.7994 on the ...,"This measures ranking discrimination, not clin..."
1,Positive-case detection,The class-weighted model recorded recall 0.766...,"Its precision was 0.3866, and its specificity ..."
2,Conservative positive predictions,The standard model recorded precision 0.6162 a...,"Its recall was only 0.1877, so many labelled p..."
3,Probability quality,The standard model had lower Brier score (0.12...,These results do not establish reliable clinic...
4,Explainability,Coefficient analysis and bootstrap refits prov...,Coefficients are not causal effects; 30 bootst...
5,Clinical interpretation,The workflow demonstrates modelling and evalua...,It is not a confirmed CKD diagnosis or a model...



Overall conclusion: The results demonstrate a trade-off between positive-case detection and precision. Further independent validation and clinical assessment are required before any clinical use.


In [9]:
# Cell 8: Final NB3 integrity check

required_objects = {
    "study_facts": study_facts,
    "nb2_test_metrics": nb2_test_metrics,
    "tradeoff_table": tradeoff_table,
    "trustworthiness_risks": trustworthiness_risks,
    "explainability_summary": explainability_summary,
    "reproducibility_checklist": reproducibility_checklist,
    "research_findings_df": research_findings_df,
}

for name, obj in required_objects.items():
    assert obj is not None, f"Missing object: {name}"

assert len(nb2_test_metrics) == 2
assert np.isclose(
    nb2_test_metrics.loc[
        nb2_test_metrics["model"] == "Standard Logistic Regression",
        "roc_auc"
    ].iloc[0],
    0.7994
)
assert np.isclose(
    nb2_test_metrics.loc[
        nb2_test_metrics["model"] == "Class-weighted Logistic Regression",
        "recall"
    ].iloc[0],
    0.7662
)

assert study_facts["heldout_test_size"] == 1654
assert len(trustworthiness_risks) == 8
assert len(research_findings_df) == 6

clinical_validation_completed = False
external_validation_completed = False
claude_ckd_demonstrator_integration = "Planned"

print("NB3 integrity checks passed.")
print("Required summary tables:", len(required_objects))
print("Held-out models documented:", len(nb2_test_metrics))
print("Trustworthiness risks:", len(trustworthiness_risks))
print("Research findings:", len(research_findings_df))
print("External validation completed:", external_validation_completed)
print("Clinical validation completed:", clinical_validation_completed)
print("Claude CKD demonstrator integration:", claude_ckd_demonstrator_integration)

NB3 integrity checks passed.
Required summary tables: 7
Held-out models documented: 2
Trustworthiness risks: 8
Research findings: 6
External validation completed: False
Clinical validation completed: False
Claude CKD demonstrator integration: Planned


9. Research Literature and Reporting Standards

9.1 Data Source and Kidney-Related Outcome Definition

This study uses data from the National Health and Nutrition Examination Survey (NHANES) 2017–March 2020 Pre-Pandemic release. NHANES provides demographic, examination, laboratory, and questionnaire data. The CDC notes that the pandemic interrupted data collection in 2020, and the combined pre-pandemic release should be used according to its published analytical guidance.

The kidney-related research outcome in this project was constructed using the 2021 CKD-EPI creatinine-based estimated glomerular filtration rate (eGFR) equation and urine albumin-to-creatinine ratio (UACR). Participants were labelled positive if eGFR was below 60 or UACR was at least 30, negative when both measurements were available and neither criterion was met, and otherwise left unlabelled.

Important limitation: This operational definition identifies kidney abnormalities in cross-sectional survey data. It does not establish a confirmed clinical diagnosis of chronic kidney disease, which requires clinical interpretation and consideration of chronicity.

9.2 Transparent Reporting of Clinical AI

The TRIPOD+AI statement provides reporting recommendations for studies developing or evaluating clinical prediction models using regression or machine-learning methods. Its emphasis on transparent reporting is relevant to documenting this project’s data, outcome definition, model development, evaluation, and limitations.

Application to this project: NB1 and NB2 document the cohort, outcome construction, predictor selection, leakage prevention, preprocessing, model evaluation, and held-out test results. NB3 consolidates these findings and identifies work that remains incomplete.

Reference:

Collins GS, Moons KGM, Dhiman P, et al. TRIPOD+AI statement: updated guidance for reporting clinical prediction models that use regression or machine learning methods. BMJ. 2024;385:e078378. https://doi.org/10.1136/bmj-2023-078378

9.3 Risk of Bias and Model Applicability

PROBAST+AI provides a framework for assessing the quality, risk of bias, and applicability of prediction-model studies using regression or artificial intelligence. It highlights the importance of evaluating participants and data sources, predictors, outcomes, and analysis methods.

Application to this project: The use of a held-out test set and leakage-aware preprocessing supports a structured evaluation, but does not by itself establish low risk of bias or clinical applicability. External validation, assessment across relevant population subgroups, and clinical evaluation remain outstanding.

Reference:

Moons KGM, Damen JA A, Kaul T, et al. PROBAST+AI: an updated quality, risk of bias, and applicability assessment tool for prediction models using regression or artificial intelligence methods. BMJ. 2025;388:e082505. https://doi.org/10.1136/bmj-2024-082505

9.4 Implications for Trustworthy Healthcare AI

The literature and reporting guidance above support a cautious approach to interpreting machine-learning results in healthcare. Discrimination, precision, recall, probability-error measures, coefficient interpretation, and stability provide different perspectives on model behaviour. No single metric establishes clinical usefulness or safety.

In this project, the standard and class-weighted logistic regression models show different precision–recall trade-offs. Coefficients and bootstrap stability summaries provide exploratory information about model behaviour, but do not establish causal relationships or prove that the model will generalise to another population.

The future healthcare demonstrator must therefore present CKD outputs as research estimates based on a cross-sectional kidney-abnormality indicator, not as a diagnosis or a substitute for professional medical assessment.

9.5 References and Official Documentation

1. CDC/National Center for Health Statistics. NHANES 2017–March 2020 Pre-Pandemic: Questionnaires, Datasets, and Related Documentation. https://wwwn.cdc.gov/nchs/nhanes/continuousnhanes/default.aspx?Cycle=2017-2020
2. National Institute of Diabetes and Digestive and Kidney Diseases (NIDDK). eGFR Equations for Adults. https://www.niddk.nih.gov/research-funding/research-programs/kidney-clinical-research-epidemiology/laboratory/glomerular-filtration-rate-equations/adults
3. Collins GS, Moons KGM, Dhiman P, et al. TRIPOD+AI statement. BMJ. 2024;385:e078378. https://doi.org/10.1136/bmj-2023-078378
4. Moons KGM, Damen JAA, Kaul T, et al. PROBAST+AI. BMJ. 2025;388:e082505. https://doi.org/10.1136/bmj-2024-082505

10. Discussion, Conclusions and Future Work

10.1 Discussion of Findings

This study examined logistic regression models for identifying a cross-sectional kidney-abnormality research indicator using selected demographic, health, examination, and laboratory-related predictors from NHANES 2017–March 2020 Pre-Pandemic data.

The standard and class-weighted logistic regression models achieved the same held-out ROC-AUC of 0.7994, while their classification behaviour differed at the default probability threshold of 0.5.

The standard model achieved higher precision (0.6162) and specificity (0.9714), but lower recall (0.1877). The class-weighted model achieved higher recall (0.7662) and F1-score (0.5139), but lower precision (0.3866) and specificity (0.7028). These results illustrate the trade-off between identifying more positive cases and limiting false-positive classifications.

The standard model also produced lower Brier score and log loss in the held-out evaluation. These results are descriptive comparisons on this dataset; they do not establish that either model is preferable for clinical use.

Coefficient analysis and bootstrap refitting provided exploratory information about model interpretation and sign stability. However, correlated predictors, the limited number of bootstrap refits, and the absence of independent validation restrict the conclusions that can be drawn from these analyses.

10.2 Study Limitations

The principal limitations are:

1. Cross-sectional outcome: The constructed outcome represents kidney abnormalities observed in survey data, not confirmed chronic kidney disease or future disease development.
2. Incomplete outcome information: Some participants could not be labelled because the required measurements were unavailable.
3. Class imbalance: Positive and negative outcomes were not equally represented, affecting classification behaviour.
4. Potential population differences: The findings may not generalise to other populations, healthcare settings, or time periods.
5. Model and explanation limitations: Logistic regression coefficients describe fitted associations, not causal effects. Bootstrap sign stability does not prove that explanations are clinically valid.
6. Validation limitations: Independent external validation, clinical validation, and prospective evaluation have not been completed.
7. Threshold limitations: The reported held-out results use a threshold of 0.5 for comparison. The held-out test set should not be used repeatedly to select a threshold.

10.3 Conclusion

This project demonstrates a structured approach to developing, evaluating, and documenting machine-learning models for a kidney-abnormality research indicator. The results highlight why accuracy or ROC-AUC alone is insufficient: precision, recall, specificity, probability-error measures, and model limitations must also be considered.

The trustworthiness assessment reinforces the importance of leakage prevention, transparent outcome construction, reproducible preprocessing, cautious interpretation, and explicit documentation of incomplete validation.

The findings are preliminary and research-oriented. They do not establish a clinically validated CKD prediction tool and must not be presented as a medical diagnosis.

10.4 Future Work

Future work should prioritise:

* Independent validation using an appropriate external dataset.
* Evaluation of performance across relevant demographic and population subgroups.
* Further assessment of calibration, threshold selection, and clinical utility using a prespecified validation strategy.
* More extensive investigation of explanation stability and correlated predictors.
* Review of data provenance, software versions, and end-to-end reproducibility.
* Careful integration of the CKD research component into the existing healthcare AI demonstrator alongside diabetes, with clear limitations, appropriate uncertainty communication, and safeguards against diagnostic claims.

10.5 Project Status

At this stage, the NB1 and NB2 model-development and evaluation results have been synthesised in NB3. The notebook documents important trustworthiness considerations, but external and clinical validation remain incomplete.

The next project milestone is to review and save the completed notebook, publish it to the project repository, and then work on the planned Claude-assisted integration of the CKD research component into the existing healthcare demonstrator. Demonstrator integration and testing must be documented separately and must not be treated as clinical validation.